# Lab 3: Search Algorithms for Warehouse Navigation

**Name:** Mehta Prem Kalpesh 

**ID:** 2024ADPS0851G

## Task 0: Understand the Search Problem

### Search problem formulation

P = (S, A, T, s0, G, c)

| Component | Specification |
|---|---|
| **State S** | Robot position (r, c), where r is the row index from the top and c is the column index from the left (both 0-indexed). S is the set of all non-obstacle cells in the 9×17 grid. |
| **Actions A** | {Up, Down, Left, Right} |
| **Transition T** | T((r,c), Up) = (r−1, c)<br>T((r,c), Down) = (r+1, c)<br>T((r,c), Left) = (r, c−1)<br>T((r,c), Right) = (r, c+1)<br>Defined only if the resulting cell lies inside the grid and is not `#`. |
| **Initial state s0** | (1, 1) |
| **Goal G** | {(7, 15)} |
| **Cost c** | c(s, a, s′) = 1 for every valid move |

### Answers

**(a) What information is necessary to specify a state?**
Only the robot's position (r, c). The warehouse layout is fixed during the search, so it is part of the problem definition and not part of the state.

**(b) What makes an action invalid?**
An action is invalid if the cell it leads to is outside the grid boundary or is an obstacle (`#`).

**(c) Is this a deterministic search problem?**
Yes. Each (state, action) pair leads to exactly one next state. There is no randomness or uncertainty in the outcome of a move.

**(d) What would constitute a solution?**
A solution is a sequence of valid actions (equivalently, a sequence of cells) that takes the robot from s0 = (1, 1) to the goal G = (7, 15), with every step being a valid transition. An *optimal* solution is one with minimum total cost, i.e. the fewest moves, since every move costs 1.

## Task 1: Plan the Agent

**1. State representation.**
A state is a tuple `(r, c)`, the robot's row and column (0-indexed). The state does not include the map, because the map is fixed.

**2. Warehouse representation.**
The warehouse is a 9×17 grid stored as a list of strings (or a list of lists of characters), where `grid[r][c]` is one of `'#'` (obstacle), `'.'` (free), `'S'` (start) or `'G'` (goal). The program scans the grid once to find the coordinates of `S` and `G`.

**3. Valid actions.**
For a state `(r, c)`, try the four moves Up `(r-1, c)`, Down `(r+1, c)`, Left `(r, c-1)`, Right `(r, c+1)`. A move is valid if the new cell is inside the grid bounds and `grid[nr][nc] != '#'`. Each valid move has cost 1.

**4. Goal test.**
The goal is reached when `state == goal`, where `goal` holds the coordinates of `G` found in step 2. The test is applied when a node is popped from the frontier, not when it is generated, so that the first goal popped is optimal.

**5. Frontier.**
The frontier is a priority queue (`heapq`) of entries `(f, g, state)`, where
- g = cost from the start to the state (number of moves so far),
- h = Manhattan distance to the goal, |r − r_G| + |c − c_G|,
- f = g + h.

The node with the smallest f is expanded next. Separately, a dictionary `best_g[state]` records the lowest g found so far for each state.

**6. Avoiding repeated expansion.**
A set `explored` records states that have already been expanded. A popped state that is in `explored` is skipped. A new neighbour is pushed only if it gives a lower g than any previously recorded `best_g`.

**7. Path reconstruction.**
A dictionary `parent[state] = previous_state` is updated whenever a better g is found. When the goal is popped, walk back from the goal to the start through `parent`, then reverse the list to get the path from S to G.

**8. Output reported on termination.**
- whether a solution was found (True/False),
- the solution path (list of `(r, c)`),
- the path length (number of moves, which is the number of cells in the path minus 1),
- the number of states expanded (counted each time a state is popped and expanded, not skipped).

If the frontier becomes empty without reaching the goal, the program reports failure with no path.

## Task 2: Ask an LLM to generate A*

**Prompt:** 
I am implementing a simple goal-based search agent in Python.

The environment is a grid given as an ASCII map. '#' is an obstacle, '.' is a free cell, 'S' is the start and 'G' is the goal (S and G are walkable). The agent can move up, down, left or right, and every move costs 1.

Implement A* search as a function that takes the map (a multi-line string) and returns the results.

Requirements:
- A state is a (row, col) tuple, 0-indexed from the top-left.
- Use Manhattan distance as the heuristic: h(n) = |row - row_G| + |col - col_G|. Pass the heuristic in as a function argument so it can be swapped later.
- The frontier is a priority queue (heapq) holding the generated-but-not-yet-expanded nodes, ordered by f = g + h. Each entry stores f, g and the state.
- Calculate g(n), h(n) and f(n) explicitly.
- Do the goal test when a node is popped from the frontier, not when it is generated.
- Avoid expanding the same state twice (an explored set, and keep the best known g for each state).
- Keep a parent pointer for each state and use it to reconstruct the path from S to G.
- Return/report: whether a solution was found, the path, the path length (number of moves), and the number of states expanded.
- If no path exists, terminate cleanly and report that no solution was found.

Keep the implementation simple, and explain the main components of the code.

Map:
```
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
```


In [ ]:
import heapq
import itertools

def manhattan_heuristic(state, goal):
    """Calculates the Manhattan distance between two (row, col) states."""
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])

def a_star_search(grid_str, heuristic=manhattan_heuristic):
    # Parse the grid to find boundaries, start, and goal
    grid = grid_str.strip().split('\n')
    rows = len(grid)
    cols = len(grid[0])
    
    start = None
    goal = None
    
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 'S':
                start = (r, c)
            elif grid[r][c] == 'G':
                goal = (r, c)
                
    if not start or not goal:
        raise ValueError("Grid must contain both 'S' and 'G'")

    # Priority queue for the frontier
    frontier = []
    # Counter serves as a tie-breaker so heapq doesn't try to compare (row, col) tuples
    counter = itertools.count() 
    
    # Initialize the starting node
    g_start = 0
    h_start = heuristic(start, goal)
    f_start = g_start + h_start
    heapq.heappush(frontier, (f_start, g_start, next(counter), start))
    
    # Trackers for path reconstruction and best known costs
    came_from = {start: None}
    best_g = {start: g_start}
    expanded_count = 0
    
    # Up, Down, Left, Right
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)] 
    
    while frontier:
        f, g, _, current = heapq.heappop(frontier)
        
        # Goal test executed strictly upon popping the node
        if current == goal:
            path = []
            curr = current
            while curr is not None:
                path.append(curr)
                curr = came_from[curr]
            path.reverse()
            
            return {
                "solution_found": True,
                "path": path,
                "path_length": len(path) - 1, # Number of moves, not nodes
                "states_expanded": expanded_count
            }
            
        # If a better path to this node was already found and processed, discard this one
        if g > best_g.get(current, float('inf')):
            continue
            
        expanded_count += 1
        
        # Generate valid neighbors
        for dr, dc in directions:
            nr, nc = current[0] + dr, current[1] + dc
            
            # Check grid bounds and obstacle collisions
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != '#':
                neighbor = (nr, nc)
                tentative_g = g + 1
                
                # Check if this new path to the neighbor is the best one found so far
                if tentative_g < best_g.get(neighbor, float('inf')):
                    best_g[neighbor] = tentative_g
                    came_from[neighbor] = current
                    
                    # Calculate new metrics
                    h = heuristic(neighbor, goal)
                    f_new = tentative_g + h
                    
                    heapq.heappush(frontier, (f_new, tentative_g, next(counter), neighbor))
                    
    # Loop exhausted with no path to the goal
    return {
        "solution_found": False,
        "path": [],
        "path_length": 0,
        "states_expanded": expanded_count
    }

# Execution on your provided map:
map_string = """
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
"""

results = a_star_search(map_string)
print(f"Solution Found: {results['solution_found']}")
print(f"Path Length: {results['path_length']}")
print(f"States Expanded: {results['states_expanded']}")
print(f"Path: {results['path']}")

Solution Found: True
Path Length: 4
States Expanded: 4
Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)]


## Task 3: Test the Generated Program

All tests were run with `a_star_search(map_string)` using the Manhattan heuristic. Expected values were worked out by hand from the map before running.

### Test 1: Original warehouse

| Measure | Result |
|---|---|
| Path found | Yes |
| Path length | 40 moves |
| States expanded | 63 |

**Path:**
(1,1) → (1,2) → (1,3) → (1,4) → (1,5) → (2,5) → (3,5) → (4,5) → (5,5) → (5,6) → … → (5,13) → (4,13) → (3,13) → (3,12) → … → (3,7) → (2,7) → (1,7) → (1,8) → … → (1,15) → (2,15) → … → (7,15)

**Validation:** An independent BFS written separately from the A* code also gives a shortest length of 40, so the path is optimal.

### Test 2: Trivial case

Map:
```
#####
#SG##
#####
```

| Measure | Result |
|---|---|
| Path found | Yes |
| Path | (1,1) → (1,2) |
| Path length | 1 |
| States expanded | 1 |

The program finds the one-step solution as expected.

### Test 3: No solution

Map:
```
#######
#S....#
###.###
#...#G#
#######
```

| Measure | Result |
|---|---|
| Path found | No |
| Path | [] (empty) |
| Path length | 0 |
| States expanded | 9 |

The program terminates and reports failure with no infinite loop. The 9 expanded states are exactly the 9 cells reachable from S (5 in row 1, 1 in row 2, 3 in row 3). This confirms that the search exhausts the reachable region and then stops.

### Test 4: Alternative paths

**Map A (two paths of equal length):**
```
#####
#S..#
#.#.#
#..G#
#####
```
Both routes (right-then-down and down-then-right) take 4 moves. The program returned the path (1,1) → (2,1) → (3,1) → (3,2) → (3,3), of length 4, which is optimal. However, because both routes have the same length, this map cannot tell a shortest path from a lucky one.

**Map B (one short path, one long path):**
```
#######
#S...G#
#.###.#
#.....#
#######
```
The short route along row 1 takes 4 moves. The long route through row 3 takes 8 moves. The expected shortest length is 4.

| Measure | Result |
|---|---|
| Path found | Yes |
| Path | (1,1) → (1,2) → (1,3) → (1,4) → (1,5) |
| Path length | 4 |
| States expanded | 4 |

The program chose the shorter route (4, not 8), so it returns a shortest path when the alternatives differ in length.

### Summary

| Test | Expected | Observed | Pass |
|---|---|---|---|
| 1. Original warehouse | A path exists; length matches independent BFS | 40 moves (BFS: 40) | ✓ |
| 2. Trivial | 1 move | 1 move | ✓ |
| 3. No solution | Failure, clean termination | Failure, 9 states expanded | ✓ |
| 4. Alternative paths | Shortest path (4) | 4 | ✓ |

## Task 4: Inspect the A* Algorithm

| Concept | Where it appears in the code |
|---|---|
| **State** | A `(row, col)` tuple: `start`, `goal`, `current`, `neighbor`. The map (`grid`, parsed from `grid_str`) is the fixed environment, not part of the state. |
| **Action** | The `directions` list: `[(-1,0), (1,0), (0,-1), (0,1)]` = Up, Down, Left, Right, looped over as `(dr, dc)`. |
| **Transition** | `nr, nc = current[0] + dr, current[1] + dc`, accepted only if `0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != '#'`. |
| **Goal test** | `if current == goal:`, applied when a node is popped from the frontier. |
| **g(n)** | `g_start = 0` for the start, `tentative_g = g + 1` for a neighbour. The best known g for each state is stored in `best_g`. |
| **h(n)** | `manhattan_heuristic(state, goal)`, called as `heuristic(start, goal)` and `heuristic(neighbor, goal)`. |
| **f(n)** | `f_start = g_start + h_start` and `f_new = tentative_g + h`. |
| **Frontier** | `frontier`, a heap managed by `heapq` that holds entries `(f, g, counter, state)`. |
| **Visited states** | No explicit explored set. `best_g` (best known cost per discovered state) and `came_from` (all discovered states), plus the check `if g > best_g.get(current, inf): continue`. |
| **Path reconstruction** | `came_from` stores each state's parent. After the goal test succeeds, the loop `while curr is not None` follows the parents back to the start, appending to `path`, and `path.reverse()` puts it in start-to-goal order. |

**(a) What data structure is used for the A\* frontier?**
A Python list used as a binary min-heap through `heapq`, which makes it a priority queue. Each entry is a tuple `(f, g, counter, state)`.

**(b) How does the program select the next state to expand?**
`heapq.heappop(frontier)` removes the entry with the smallest tuple, so the smallest f. If f ties, the smaller g wins, and then the smaller `counter` (the earlier-inserted entry). The `counter` also stops `heapq` from comparing state tuples directly.

**(c) Where is the heuristic calculated?**
In `manhattan_heuristic(state, goal)`, which returns |r − r_G| + |c − c_G|. It is called once for the start state and once for every neighbour that is pushed to the frontier.

**(d) Does the program explicitly calculate f(n) = g(n) + h(n)?**
Yes. It computes `f_start = g_start + h_start` for the start state and `f_new = tentative_g + h` for each neighbour. The value is stored as the first element of the heap entry and determines the expansion order.

**(e) How does the program prevent unnecessary repeated exploration?**
Two mechanisms work together:
1. A neighbour is pushed only if `tentative_g < best_g.get(neighbor, inf)`, i.e. only when a strictly cheaper path to it has been found.
2. When an entry is popped, `if g > best_g.get(current, inf): continue` discards stale entries, which are older, more expensive entries for a state that has since been reached more cheaply. Skipped entries are not counted as expansions.

## Task 5: Compare A* with Blind Search

**Prompt:** 
I am implementing a simple goal-based search agent in Python.

The environment is a grid given as an ASCII map. '#' is an obstacle, '.' is a free cell, 'S' is the start and 'G' is the goal (S and G are walkable). The agent can move up, down, left or right, and every move costs 1.

Implement BFS as a function `bfs_search(grid_str)` that takes the map (a multi-line string) and returns a dictionary.

Requirements:
- A state is a (row, col) tuple, 0-indexed from the top-left.
- The frontier is a FIFO queue (collections.deque) holding nodes that have been discovered but not yet expanded.
- Expand neighbours in the order Up, Down, Left, Right, i.e. directions (-1,0), (1,0), (0,-1), (0,1).
- Mark a state as discovered when it is added to the queue, so no state is queued twice.
- Do the goal test when a node is popped from the frontier, not when it is generated.
- Count a state as expanded only when its neighbours are generated. The goal node, which is popped and returned, is not counted.
- Keep a parent pointer for each state and use it to reconstruct the path from S to G.
- Return a dictionary with keys: "solution_found" (bool), "path" (list of (row, col)), "path_length" (number of moves), "states_expanded" (int).
- If no path exists, terminate cleanly and return solution_found=False, an empty path, path_length 0.

Keep the implementation simple, and explain the main components of the code.

Map:
```
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#.............#.#
#.###########.#.#
#.............#G#
#################
```


In [8]:
import collections

def bfs_search(grid_str):
    # Parse the grid from the string
    grid = grid_str.strip().split('\n')
    rows = len(grid)
    cols = len(grid[0]) if rows > 0 else 0
    
    start = None
    goal = None
    
    # Locate Start (S) and Goal (G)
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 'S':
                start = (r, c)
            elif grid[r][c] == 'G':
                goal = (r, c)
                
    if not start or not goal:
        return {"solution_found": False, "path": [], "path_length": 0, "states_expanded": 0}

    # Frontier is a FIFO queue
    frontier = collections.deque([start])
    
    # The parent dictionary keeps track of where we came from to reconstruct the path.
    # It also serves as our 'discovered' set to prevent adding nodes to the queue twice.
    parent = {start: None}
    states_expanded = 0
    
    # Up (-1, 0), Down (1, 0), Left (0, -1), Right (0, 1)
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    while frontier:
        # Pop from the left to maintain FIFO (Breadth-First) order
        current = frontier.popleft()
        
        # 1. Goal test when a node is popped, not when it is generated
        if current == goal:
            # Reconstruct the path by backtracking from the goal
            path = []
            curr = current
            while curr is not None:
                path.append(curr)
                curr = parent[curr]
            path.reverse()
            
            return {
                "solution_found": True,
                "path": path,
                "path_length": len(path) - 1,
                "states_expanded": states_expanded
            }
            
        # 2. Count a state as expanded only right before generating its neighbors
        states_expanded += 1
        
        r, c = current
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            
            # Check boundaries and obstacles
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != '#':
                # 3. Mark as discovered when added to the queue
                if (nr, nc) not in parent:
                    parent[(nr, nc)] = current
                    frontier.append((nr, nc))
                    
    # Clean termination if the queue empties and no path exists
    return {
        "solution_found": False,
        "path": [],
        "path_length": 0,
        "states_expanded": states_expanded
    }

# Example Usage:
if __name__ == "__main__":
    map_str = """
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
"""
    result = bfs_search(map_str)
    print(f"Found: {result['solution_found']}")
    print(f"Path Length: {result['path_length']}")
    print(f"States Expanded: {result['states_expanded']}")
    print(f"Path: {result['path']}")

Found: True
Path Length: 40
States Expanded: 63
Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]


## Task 5: Compare A* with Blind Search

Both algorithms were run on the original, unchanged warehouse map. They use the same neighbour order (Up, Down, Left, Right) and the same rule for counting expanded states (the goal node is not counted).

| Measure | BFS | A* (Manhattan) |
|---|---|---|
| Solution found | Yes | Yes |
| Path length | 40 | 40 |
| States expanded | 63 | 63 |

The two algorithms returned the identical path.

**(a) Did both algorithms find a solution?**
Yes. Both found a path from S = (1,1) to G = (7,15).

**(b) Did they find paths of the same length?**
Yes, both found a path of 40 moves. This is expected. BFS finds the shortest path when all moves cost 1, and A* with an admissible heuristic also finds an optimal path.

**(c) Which algorithm expanded fewer states?**
Neither. Both expanded 63 states, so on this map A* gave no saving over BFS.

**(d) Why might A\* expand fewer states?**
In general, A* uses the heuristic h(n) to prefer states that appear closer to the goal. Where there are many open routes, it can ignore regions that lead away from G, while BFS expands every state in order of distance from S. A* is guaranteed to expand every state with f(n) = g(n) + h(n) < C*, where C* is the optimal cost (here 40).

This warehouse is a winding maze with few branches, and the true route from S to G moves away from G at several points before returning. The Manhattan distance ignores the walls, so it underestimates the real distance by a large margin. As a result f(n) stays below 40 for almost every reachable state, so A* has to expand nearly the same states as BFS. The heuristic gives A* information, but in this map that information is weak, so it does not prune anything. A* is better than blind search only when its heuristic is informative for the environment.

## Task 6: Investigate the Heuristic

**Why Manhattan distance is appropriate.** The robot moves only up, down, left and right with cost 1, so any path from (r, c) to the goal needs at least |r − r_G| + |c − c_G| moves. Walls can only make it longer, never shorter. So h(n) ≤ h*(n): the heuristic is admissible. It is also consistent.

### Experiment 1: the warehouse

| Heuristic | Found | Path length | States expanded |
|---|---|---|---|
| Manhattan (baseline) | Yes | 40 | 63 |
| h(n) = 0 | Yes | 40 | 63 |
| Euclidean | Yes | 40 | 63 |
| 2 × Manhattan | Yes | 40 | 66 |

On the warehouse the heuristics barely differ. The map is a winding maze with few branches, so no heuristic can prune much. All four find an optimal path.

### Experiment 2: an open grid

An 8×10 grid with no obstacles, S at the top-left and G at the bottom-right.

| Heuristic | Found | Path length | States expanded |
|---|---|---|---|
| Manhattan | Yes | 16 | 79 |
| h(n) = 0 | Yes | 16 | 79 |
| Euclidean | Yes | 16 | 79 |
| 2 × Manhattan | Yes | 16 | 16 |
| BFS | Yes | 16 | 79 |

Manhattan expands as many states as BFS, because many cells have the same f = 16 and the heap breaks ties in favour of smaller g. The inflated heuristic expands only 16 states, since it is greedy.

### Experiment 3: a map where the aggressive heuristic loses optimality

```
##########
#S.#.....#
#..#..#.##
#.#....#.#
#..#.#...#
##.......#
#......#G#
##########
```

| Heuristic | Found | Path length | States expanded |
|---|---|---|---|
| Manhattan | Yes | 12 | 19 |
| 2 × Manhattan | Yes | **14** | 14 |
| BFS | Yes | 12 | 32 |

### Answers

1. **h(n) = 0.** Then f = g and A* becomes uniform-cost search, equivalent to BFS with unit costs. It is still optimal, but it has no information about the goal, so it expands as many states as BFS (63 on the warehouse, 79 on the open grid).
2. **Euclidean.** It is admissible (it never exceeds Manhattan, which is a lower bound), so it stays optimal. It is weaker than Manhattan on a 4-connected grid, since it is always ≤ Manhattan. It gave the same results here.
3. **2 × Manhattan.** This is no longer admissible: 2h can exceed the true remaining cost. A* becomes greedier. It expanded fewer states on the open grid (16 against 79) and the suboptimal map (14 against 19), but returned a **longer** path on the third map (14 against 12). On the warehouse it expanded slightly more (66 against 63).

**Conclusion.** An admissible heuristic guarantees an optimal path, and a closer-to-h* estimate expands fewer states. An over-optimistic or too aggressive heuristic can speed up the search but loses the guarantee of optimality.

## Task 7: Evaluate the LLM-Generated Agent

**1. What parts of the generated code were correct immediately?**
The A* and BFS code both worked on the first run. Parsing the map, finding S and G, the bounds and obstacle check, goal testing on pop, path reconstruction through parent pointers, the position of the `expanded_count` increment, and clean termination with no solution were all correct. I did not assume this. I checked each part with Tests 1 to 4 and compared the optimal path length against an independent BFS.

**2. Did you find any bugs or design problems?**
I found no bugs in the generated code. I found design problems elsewhere:
- My own design had mistakes: the goal coordinates in Task 0 were wrong, and my first description of the frontier was wrong.
- My first prompt was missing the failure case, the coordinate convention and a reusable function.
- My first Test 4 map had two equal-length paths, so it could not show optimality.
- The code uses `best_g` and a stale-entry check instead of the explored set in my design.
- The heap order `(f, g, counter)` breaks ties toward smaller g, which makes A* expand as many states as BFS on an open grid.

**3. How did you discover those problems?**
By checking coordinates against the map, by reviewing my design and prompts before running anything, by reasoning about what each test could and could not show, by inspecting the code in Task 4, and by the heuristic experiments in Task 6.

**4. Did the LLM use terminology or data structures that you did not understand?**
Mostly no. The names were clear and the comments explained the steps. The less obvious points were the `counter` in the heap entries (it stops `heapq` from comparing state tuples when f and g are equal) and the use of `best_g` with a stale-entry check, which I first read as a visited set.

**5. Did you modify the LLM-generated code?**
No, I did not change the algorithm code. I changed the prompts (after review), the input map in the BFS run, and the heuristic function passed in for Task 6.

**6. Which tests were most useful?**
Test 3 (no solution) because it checks termination, which is easy to get wrong and would be an infinite loop if the program were wrong. Test 4B (one short path, one long path) because it is the only test that shows the returned path is shortest. The independent BFS comparison on the original warehouse was also useful.

**7. Could you have trusted the program without testing it?**
No. The code looked correct, but looking correct is not the same as being validated. Testing showed that termination and optimality were right, and the experiments showed behaviour I would not have noticed by reading, such as the tie-breaking effect. Mistakes in my own design and in the input map would also have gone unnoticed without checking.

**8. What did you understand about A\* that you did not understand before implementing it?**
- A* keeps the best known g for each state, and may hold several entries for the same state in the frontier, discarding stale ones when they are popped.
- The goal test must happen when a node is popped, not when it is generated, so that the first path found is optimal.
- The heuristic only estimates the remaining cost, and its quality decides how much A* saves. On the winding warehouse, A* expanded the same 63 states as BFS.
- An admissible heuristic keeps the path optimal. An over-aggressive one (2 × Manhattan) can return a longer path while expanding fewer states.
- Details such as tie-breaking can change A*'s behaviour a lot.

## Final Reflection

**1. Why is it important to formulate the search problem before writing the search algorithm?**
The formulation defines what the program must do: what a state is, which moves are legal, what the goal is, and what a solution and its cost are. Without it, there is nothing to compare the code against, so "it looks right" is the only test available. In this lab, an early mistake in the goal coordinates (I had written (16,16) where the goal is at (7,15)) would not have crashed anything. It would have produced a wrong heuristic and plausible-looking results. Having a written formulation also gave the LLM a precise specification, so it implemented my design instead of inventing details, and it gave me test cases whose correct answers I could work out by hand.

**2. In what sense is A\* an "informed" search algorithm?**
BFS decides what to expand next using only the order in which states were discovered. A* uses extra knowledge about the problem: the heuristic h(n), an estimate of the remaining cost from n to the goal. It expands the state with the lowest f(n) = g(n) + h(n), so it combines the cost already paid with a guess about the cost still to come. "Informed" does not guarantee that A* does less work, because that depends on how good the information is. On the winding warehouse, A* expanded the same 63 states as BFS, since Manhattan distance ignores the walls and so tells the search very little.

**3. Why does the choice of heuristic matter?**
The heuristic affects both correctness and efficiency. If it is admissible (never above the true remaining cost), A* returns an optimal path. If it is inadmissible, as 2 × Manhattan was in my experiment, A* can return a longer path: on one map it returned 14 moves where the optimum was 12, even though it expanded fewer states. Efficiency depends on how close the heuristic is to the true cost. With h = 0, A* reduces to uniform-cost search and expands as much as BFS. On an open grid, 2 × Manhattan expanded 16 states against 79 for Manhattan. A good heuristic is admissible and as close to the true cost as possible.

**4. What did the LLM contribute to the engineering process?**
The LLM translated my design into working code for both A* and BFS, and explained the main components. The code was correct on the first run: the parsing, the bounds checks, the goal test on pop, the path reconstruction and the clean failure case. It also made design decisions I had not specified: the `best_g` dictionary with a stale-entry check instead of an explored set, a tie-breaking counter, and the order `(f, g, counter)` in the heap. That last choice turned out to matter, because it makes A* behave like BFS on open grids. The formulation, the design, the tests and the interpretation of the results were mine.

**5. What could go wrong if an engineer simply accepted LLM-generated code without testing it?**
The program could return a path that looks plausible but is not shortest, loop forever on a map with no solution, or fail on edge cases such as a start next to the goal. In this lab the generated code was correct, but I would not have known that without testing, and testing also showed behaviour I could not have seen from reading: the tie-breaking effect, the lost optimality of an inflated heuristic, and the fact that my first Test 4 map could not detect a non-optimal path. Wrong inputs are also easy to miss, such as the mistyped map in a prompt, which gave a result close to the right one but not the same. Accepting code unchecked means shipping plausible output as if it were validated.